## Prepare the raw Largaespada lab MPNST data

The delivered files, under `data/largaespada/raw_data/`, don't quite match
what `1.load_and_assess_gene_overlap.py`, `2.format_and_scale_data.py`, and
`4.apply_biobombe_ensemble.py` expect:
- `gene_effects.csv` is already samples-as-rows, genes-as-columns (DepMap's
  convention) and already rescored against the correct DepMap release
  BioBombe was trained on — but its sample identifier column is named
  "cell_line_name", not "ModelID".
- `largaespada_CRISPR_samples.csv`'s sample identifier column is named
  "sample", not "ModelID".
- A handful of gene columns use gene symbols HGNC has since renamed.
  BioBombe's own gene set (`3.run-biobombe/data/trained_gene_order.parquet`)
  is left exactly as it was trained on, out-of-date symbols and all; this
  only harmonizes the new data's symbols to match it.

  Resolution works backward from the trained set, not forward from the
  raw data: for each trained gene missing under its current symbol, ask
  mygene.info for that gene's own former names/aliases, then check for an
  exact match among the raw column names. Going the other way (resolving
  each raw column's symbol independently) hits genuinely ambiguous
  aliases shared by unrelated genes (e.g. "STRA13" names both CENPX and
  the unrelated BHLHE40) with no principled way to disambiguate; asking
  "what were CENPX's former names" instead has exactly one right answer.

This writes `NF1_data.parquet` and `NF1_metadata.parquet` directly under
`data/largaespada/`, the exact filenames those scripts already hardcode,
so nothing downstream needs to change. It also writes
`data/largaespada/results/genes_missing_in_raw_data.parquet`, recording
every trained gene with no usable value anywhere in this data (as of the
last time this ran) — either because no column matches it under any known
name, or because its column exists but is entirely missing (NaN) for
every sample. Both cases most likely mean this screen's guide library
simply didn't cover that gene, not a resolution failure; either way,
`2.format_and_scale_data` imputes them with the training-split mean like
any other gene missing from a new dataset.

Run this script with `8.apply-biobombe-new-data/` as the working
directory, once, whenever the raw files are replaced.

In [1]:
import pathlib

import pandas as pd
import requests

largaespada_dir = pathlib.Path("data/largaespada")
raw_dir = largaespada_dir / "raw_data"
results_dir = largaespada_dir / "results"
results_dir.mkdir(parents=True, exist_ok=True)

In [2]:
gene_effects = pd.read_csv(raw_dir / "gene_effects.csv")
gene_effects = gene_effects.rename(columns={"cell_line_name": "ModelID"})
print(f"Loaded raw data: {gene_effects.shape[0]} samples, {gene_effects.shape[1] - 1} genes")

Loaded raw data: 8 samples, 19168 genes


## Harmonize gene symbols to BioBombe's trained convention

Only the trained gene set matters here: nothing outside it is used by the
ensemble regardless of what it's named, so resolution targets exactly the
genes BioBombe was trained on, not DepMap's full gene set.

In [3]:
trained_gene_dict = pd.read_parquet(pathlib.Path("../3.run-biobombe/data/trained_gene_order.parquet").resolve())
trained_symbols = set(trained_gene_dict["symbol_id"])

raw_gene_columns = set(gene_effects.columns) - {"ModelID"}
missing_trained_symbols = sorted(trained_symbols - raw_gene_columns)
print(f"{len(trained_symbols) - len(missing_trained_symbols)} / {len(trained_symbols)} trained genes "
      f"already present under their current symbol; looking up aliases for the other {len(missing_trained_symbols)}")

2701 / 2718 trained genes already present under their current symbol; looking up aliases for the other 17


In [4]:
def fetch_aliases(symbols, batch_size=500):
    """
    Fetch each gene's own known aliases/former symbols from mygene.info.

    Returns
    -------
    dict
        Maps each input (current) symbol to its list of aliases (possibly empty).
    """
    aliases_by_symbol = {}
    for start in range(0, len(symbols), batch_size):
        batch = symbols[start:start + batch_size]
        response = requests.post(
            "https://mygene.info/v3/query",
            data={
                "q": ",".join(batch),
                "scopes": "symbol",
                "species": "human",
                "fields": "alias",
            },
            timeout=60,
        )
        response.raise_for_status()
        for hit in response.json():
            symbol = hit["query"]
            alias = hit.get("alias", [])
            aliases_by_symbol.setdefault(symbol, []).extend(
                [alias] if isinstance(alias, str) else alias
            )
    return aliases_by_symbol

In [5]:
aliases_by_symbol = fetch_aliases(missing_trained_symbols) if missing_trained_symbols else {}

rename_map = {}
still_missing = []
for symbol in missing_trained_symbols:
    match = next((a for a in aliases_by_symbol.get(symbol, []) if a in raw_gene_columns), None)
    if match is not None:
        rename_map[match] = symbol
    else:
        still_missing.append(symbol)

print(f"Resolved {len(rename_map)} / {len(missing_trained_symbols)} missing trained genes via a former alias")
if still_missing:
    print(f"Still missing after alias lookup: {still_missing}")

Resolved 0 / 17 missing trained genes via a former alias
Still missing after alias lookup: ['C7orf50', 'CCDC115', 'CENPJ', 'CIR1', 'FAM136A', 'FAM210A', 'FAM83H', 'FAM98B', 'N6AMT1', 'NCL', 'NEPRO', 'PRPF4B', 'SPATA5', 'SPATA5L1', 'TBRG1', 'TBRG4', 'TMEM199']


In [6]:
gene_effects = gene_effects.rename(columns=rename_map)

# A trained gene can also be missing in a second way that the alias lookup
# above can't see: its column exists (whether originally or after renaming)
# but every value in it is NaN. gene_effects.csv carries a placeholder column
# for genes outside this screen's guide library rather than omitting them,
# so this check only looks among the trained genes that do have a column.
present_trained_symbols = [g for g in trained_symbols if g in gene_effects.columns]
all_nan_trained = [g for g in present_trained_symbols if gene_effects[g].isna().all()]
print(f"{len(all_nan_trained)} / {len(present_trained_symbols)} trained genes with a matching column "
      "have no value for any sample (most likely outside this screen's guide library)")

missing_genes_df = pd.concat([
    pd.DataFrame({"trained_gene": still_missing, "reason": "no_matching_column"}),
    pd.DataFrame({"trained_gene": all_nan_trained, "reason": "column_present_but_all_values_missing"}),
], ignore_index=True)

gene_effects.to_parquet(largaespada_dir / "NF1_data.parquet", index=False)
print(f"Wrote NF1_data.parquet: {gene_effects.shape[0]} samples, {gene_effects.shape[1] - 1} genes")

sample_metadata = pd.read_csv(raw_dir / "largaespada_CRISPR_samples.csv")
sample_metadata = sample_metadata.rename(columns={"sample": "ModelID"})
sample_metadata.to_parquet(largaespada_dir / "NF1_metadata.parquet", index=False)
print(f"Wrote NF1_metadata.parquet: {sample_metadata.shape[0]} samples, "
      f"columns {list(sample_metadata.columns)}")

missing_genes_df.to_parquet(results_dir / "genes_missing_in_raw_data.parquet", index=False)
print(f"Wrote genes_missing_in_raw_data.parquet: {len(missing_genes_df)} / {len(trained_symbols)} trained genes "
      "have no usable value anywhere in the raw data")

14 / 2701 trained genes with a matching column have no value for any sample (most likely outside this screen's guide library)


Wrote NF1_data.parquet: 8 samples, 19168 genes
Wrote NF1_metadata.parquet: 8 samples, columns ['ModelID', 'SUZ12', 'NF1', 'type']
Wrote genes_missing_in_raw_data.parquet: 31 / 2718 trained genes have no usable value anywhere in the raw data
